# exp000_eda

Initial EDA for BirdCLEF 2026.

Goals:
- confirm dataset shape
- inspect target labels
- check class imbalance
- listen to a few recordings
- compare waveform and spectrogram structure
- capture domain notes that may matter later

In [ ]:
from pathlib import Path
import ast
import random

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
random.seed(42)

ROOT = Path.cwd().resolve()
if ROOT.name == "exp000_eda":
    ROOT = ROOT.parents[1]
elif not (ROOT / "train.csv").exists():
    raise FileNotFoundError("Run this notebook from the repository root or experiments/exp000_eda.")

ROOT

In [ ]:
train = pd.read_csv(ROOT / "train.csv")
taxonomy = pd.read_csv(ROOT / "taxonomy.csv")
soundscape_labels = pd.read_csv(ROOT / "train_soundscapes_labels.csv")
sample_submission = pd.read_csv(ROOT / "sample_submission.csv")

target_labels = pd.Index(sample_submission.columns[1:], name="primary_label")

print("train rows:", len(train))
print("taxonomy rows:", len(taxonomy))
print("soundscape label rows:", len(soundscape_labels))
print("submission targets:", len(target_labels))

In [ ]:
train.head()

In [ ]:
summary = pd.DataFrame({
    "train_unique_primary_label": [train["primary_label"].nunique()],
    "submission_targets": [len(target_labels)],
    "train_not_in_submission": [len(set(train["primary_label"]) - set(target_labels))],
    "submission_not_in_train": [len(set(target_labels) - set(train["primary_label"]))],
})
summary

In [ ]:
label_counts = train["primary_label"].value_counts().rename_axis("primary_label").reset_index(name="count")
label_counts.head(20)

In [ ]:
ax = sns.barplot(data=label_counts.head(20), x="primary_label", y="count", color="#3b7ddd")
ax.set_title("Top 20 primary_label counts")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()

In [ ]:
train[["type", "rating"]].describe(include="all")

In [ ]:
secondary_count = (
    train["secondary_labels"]
    .fillna("[]")
    .map(lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0)
)

secondary_count.describe()

## Audio inspection helpers

In [ ]:
taxonomy_lookup = taxonomy.set_index("primary_label")


def sample_recording(label: str, index: int = 0) -> pd.Series:
    rows = train.loc[train["primary_label"] == label].reset_index(drop=True)
    if rows.empty:
        raise ValueError(f"Unknown label: {label}")
    return rows.iloc[index % len(rows)]


def audio_path(row: pd.Series) -> Path:
    return ROOT / "train_audio" / row["filename"]


def load_audio(path: Path, offset: float = 0.0, duration: float = 10.0):
    info = sf.info(path)
    start_frame = int(offset * info.samplerate)
    num_frames = int(duration * info.samplerate)
    y, sr = sf.read(path, start=start_frame, frames=num_frames, dtype="float32")
    if getattr(y, "ndim", 1) == 2:
        y = y.mean(axis=1)
    return y, sr


def inspect_recording(label: str, index: int = 0, offset: float = 0.0, duration: float = 10.0) -> pd.Series:
    row = sample_recording(label, index=index)
    path = audio_path(row)
    y, sr = load_audio(path, offset=offset, duration=duration)

    display(row[["primary_label", "scientific_name", "common_name", "type", "rating", "secondary_labels"]])
    print(path)
    display(ipd.Audio(y, rate=sr))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color="#2d6a4f")
    axes[0].set_title(f"Waveform: {label}")

    spec = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128), ref=np.max)
    librosa.display.specshow(spec, sr=sr, x_axis="time", y_axis="mel", ax=axes[1], cmap="magma")
    axes[1].set_title(f"Mel spectrogram: {label}")
    plt.tight_layout()
    return row

In [ ]:
inspect_recording("banana", index=0, duration=8.0)

In [ ]:
inspect_recording(label_counts.iloc[0]["primary_label"], index=0, duration=8.0)

## Domain notes

Use `docs/DOMAIN_NOTES.md` to track species-level behavioral notes that may affect modeling choices, for example:
- male vs female differences
- song vs call vs alarm behavior
- time-of-day effects
- habitat-specific background noise

## Notes

Record the main findings in `notes.md` and promote reusable logic into `src/` only after it appears more than once.